# Local-SDFT: few-step Colab demo (230M)

Self-Distillation Fine-Tuning (**SDFT**) is on-policy self-distillation: show a few gold demos, let the model answer, then LoRA-train on those self-generated targets — instead of plain SFT on off-policy gold text.

**This notebook is standalone** — no `git clone`, no Local-SDFT package import.

| Arm | What it does |
|-----|----------------|
| **ZS** | Zero-shot base |
| **ICL** | Few-shot in-context (demos from the train slice) |
| **CoT** | Chain-of-thought cue |
| **gold SFT** | LoRA on gold `output` (`batch_size=1`, `STEP_COUNT` steps) |
| **SDFT** | Teacher generate → LoRA on self-distilled targets (same step budget) |

Paper: [Shenfeld et al., 2026](https://self-distillation.github.io/SDFT) ([arXiv](https://arxiv.org/abs/2601.19897)).  
Base: [Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M).

**Demo protocol (swept locally):** shuffle `yahma/alpaca-cleaned` with fixed `SEED`, take `data[:STEP_COUNT]` as the train/eval set. Train with `per_device_train_batch_size=1` for exactly `STEP_COUNT` optimizer steps. Score with **fast heuristics** (refusal rate, length, instruction reward, pairwise win counts) + a side-by-side table — **not** a full AlpacaEval / Qwen judge pass.

**Keep-alive:** one `from_pretrained` covers base ZS/ICL/CoT → SDFT teacher → LoRA train → batched eval. Gold SFT reuses the same base after LoRA unload (no second download). GPU is held for the optional full remaining-data stage below (released after that, or run `release_gpu()` if you skip it).

**Optional full stage (end of notebook):** train on `data[STEP_COUNT:]` from the same seeded shuffle with max batch size + heuristic eval — no Qwen judge.


## Setup

- **Colab**: Runtime → GPU (T4 is fine).
- Installs deps and uninstalls stale `torchao` so peft LoRA does not hit a version gate.


In [ ]:
%pip install -q "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" "pyyaml>=6.0" "huggingface_hub>=0.23"
%pip uninstall -y torchao

print("setup ok — few-step SDFT demo (heuristic metrics; no AE2 judge)")


## Helpers (inlined)

Hard-coded winners from a local MPS sweep (`seed × STEP_COUNT`, then `lr / LoRA r / teacher shots`).  
Metric: heuristic `instruction_reward` + pairwise win counts on the demo slice (no Qwen judge).


In [ ]:
import gc
import json
import random
import re
from pathlib import Path
from typing import Any, Callable

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig as PeftLoraConfig
from peft import PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

# ---------------------------------------------------------------------------
# Hard-coded demo winners (local sweep on MPS; heuristic metric)
# ---------------------------------------------------------------------------
MODEL_NAME = "LiquidAI/LFM2.5-230M"

SEED = 13
STEP_COUNT = 32          # train rows = optimizer steps (batch_size=1)
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM = 1
TRAIN_LR = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET = r".*self_attn\.(q|k|v|out)_proj"
TEACHER_NUM_SHOTS = 2
TEACHER_BATCH_SIZE = 8
TRAIN_MAX_LENGTH = 512
MAX_NEW_TOKENS = 128
MIN_RESPONSE_CHARS = 1

ICL_K = 3
COT_LINE = "Let's think step by step."

# Eval generate: start large, auto-backoff on OOM (T4-friendly)
EVAL_BATCH_CANDIDATES = (32, 16, 8, 4, 2, 1)

# Optional full remaining-data stage (after demo). Cap rows for Colab time;
# set FULL_MAX_ROWS = None to use all of data[STEP_COUNT:].
FULL_BATCH_SIZE = 16
FULL_BATCH_CANDIDATES = (32, 16, 8, 4, 2, 1)  # OOM backoff for train
FULL_EPOCHS = 1
FULL_MAX_ROWS = 512  # remaining slice length after STEP_COUNT prefix
FULL_TEACHER_BATCH = 8
FULL_OUT = Path("outputs/colab_sdft_full")
FULL_GOLD_PATH = FULL_OUT / "gold.jsonl"
FULL_SDFT_PATH = FULL_OUT / "sdft.jsonl"
FULL_SFT_ADAPTER = FULL_OUT / "adapter-sft-gold"
FULL_SDFT_ADAPTER = FULL_OUT / "adapter-sdft"

OUT_ROOT = Path("outputs/colab_sdft_demo")
GOLD_PATH = OUT_ROOT / "gold.jsonl"
SDFT_PATH = OUT_ROOT / "sdft.jsonl"
SFT_ADAPTER = OUT_ROOT / "adapter-sft-gold"
SDFT_ADAPTER = OUT_ROOT / "adapter-sdft"

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")


def release_gpu():
    """Only call at end of the demo (or if you deliberately tear down)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def load_tokenizer(name: str = MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_policy(name: str = MODEL_NAME):
    """Single base load for the whole demo / full stage machine."""
    use_cuda = torch.cuda.is_available()
    # Prefer fp16 on CUDA (T4-friendly); fp32 elsewhere for MPS/CPU stability.
    dtype = torch.float16 if use_cuda else torch.float32
    kwargs: dict[str, Any] = {"dtype": dtype}
    if use_cuda:
        kwargs["device_map"] = "auto"
        return AutoModelForCausalLM.from_pretrained(name, **kwargs)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return AutoModelForCausalLM.from_pretrained(name, **kwargs).to("mps")
    return AutoModelForCausalLM.from_pretrained(name, **kwargs)


def to_model_device(batch, model):
    device = next(model.parameters()).device
    if hasattr(batch, "to"):
        return batch.to(device)
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def load_demo_slice(step_count: int = STEP_COUNT, seed: int = SEED) -> list[dict]:
    """Shuffle alpaca-cleaned with seed; return data[:step_count]."""
    return load_seeded_rows(skip=0, limit=step_count, seed=seed)


def load_seeded_rows(
    *,
    skip: int = 0,
    limit: int | None = None,
    seed: int = SEED,
) -> list[dict]:
    """Shuffle alpaca-cleaned with seed; return data[skip:skip+limit] (or to end)."""
    ds = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=seed)
    out: list[dict] = []
    seen = 0
    for row in ds:
        parts = []
        for f in ("instruction", "input"):
            v = str(row.get(f) or "").strip()
            if v:
                parts.append(v)
        prompt = "\n\n".join(parts)
        response = str(row.get("output") or "").strip()
        if not (prompt and response):
            continue
        if seen < skip:
            seen += 1
            continue
        out.append({"prompt": prompt, "response": response})
        if limit is not None and len(out) >= limit:
            break
    return out


def load_remaining_slice(
    step_count: int = STEP_COUNT,
    seed: int = SEED,
    max_rows: int | None = FULL_MAX_ROWS,
) -> list[dict]:
    """Same seeded shuffle as the demo; return data[step_count:] (optionally capped)."""
    return load_seeded_rows(skip=step_count, limit=max_rows, seed=seed)


def sample_fewshots(examples, exclude_idx, num_shots, rng):
    pool = list(range(len(examples)))
    pool.remove(exclude_idx)
    idxs = rng.sample(pool, min(num_shots, len(pool)))
    return [examples[i] for i in idxs]


def build_teacher_messages(fewshots, prompt):
    messages = []
    for shot in fewshots:
        messages.append({"role": "user", "content": shot["prompt"]})
        messages.append({"role": "assistant", "content": shot["response"]})
    messages.append({"role": "user", "content": prompt})
    return messages


def build_eval_messages(instruction, *, few_shots=None, cot=False):
    messages = []
    for shot in few_shots or []:
        messages.append({"role": "user", "content": shot["prompt"]})
        messages.append({"role": "assistant", "content": shot["response"]})
    user = instruction.strip()
    if cot and not user.endswith(COT_LINE):
        user = f"{user}\n\n{COT_LINE}" if user else COT_LINE
    messages.append({"role": "user", "content": user})
    return messages


def pick_icl_demos(train_ex, k=ICL_K):
    return train_ex[: min(k, len(train_ex))]


def instruction_reward(text: str, gold: str = "") -> float:
    if not text:
        return -1.0
    score = 0.0
    if REFUSAL_RE.search(text):
        score -= 1.0
    else:
        score += 0.5
    n = len(text)
    if 40 <= n <= 1200:
        score += 0.5
    elif n < 20:
        score -= 0.5
    elif n > 2000:
        score -= 0.25
    if gold:
        ta, tb = set(text.lower().split()), set(gold.lower().split())
        if ta and tb:
            score += 1.5 * len(ta & tb) / len(ta | tb)
    return float(score)


def _is_oom(exc: BaseException) -> bool:
    msg = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in msg or "oom" in msg


@torch.inference_mode()
def generate_batched(
    model,
    tokenizer,
    prompts: list[str],
    *,
    messages_fn: Callable[[str], list[dict]],
    max_new_tokens: int = MAX_NEW_TOKENS,
    label: str = "gen",
    batch_sizes: tuple[int, ...] = EVAL_BATCH_CANDIDATES,
) -> list[str]:
    """Max-batch generate with OOM auto-backoff. Model stays resident."""
    tokenizer.padding_side = "left"
    model.eval()
    outs: list[str | None] = [None] * len(prompts)
    start = 0
    batch_i = 0
    while start < len(prompts):
        bs = batch_sizes[min(batch_i, len(batch_sizes) - 1)]
        bs = min(bs, len(prompts) - start)
        batch = prompts[start : start + bs]
        texts = [
            tokenizer.apply_chat_template(
                messages_fn(p), tokenize=False, add_generation_prompt=True
            )
            for p in batch
        ]
        try:
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False)
            enc = to_model_device(enc, model)
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            new = out[:, enc["input_ids"].shape[1] :]
            decoded = tokenizer.batch_decode(new, skip_special_tokens=True)
            for i, text in enumerate(decoded):
                outs[start + i] = text.strip()
            print(f"  {label} {start + bs}/{len(prompts)} (bs={bs})", flush=True)
            start += bs
            # after success, try larger batches again from the top candidate
            batch_i = 0
        except Exception as exc:
            if _is_oom(exc) and batch_i + 1 < len(batch_sizes):
                batch_i += 1
                print(f"  {label} OOM at bs={bs} → retry bs={batch_sizes[batch_i]}", flush=True)
                release_gpu()
                continue
            raise
    return [o or "" for o in outs]


@torch.inference_mode()
def teacher_generate_resident(
    model,
    tokenizer,
    examples: list[dict],
    *,
    num_shots: int = TEACHER_NUM_SHOTS,
    batch_size: int = TEACHER_BATCH_SIZE,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> list[str | None]:
    """SDFT teacher pass on the resident base (no reload)."""
    tokenizer.padding_side = "left"
    model.eval()
    rng = random.Random(SEED)
    results: list[str | None] = [None] * len(examples)
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                build_teacher_messages(
                    sample_fewshots(examples, start + i, num_shots, rng),
                    ex["prompt"],
                ),
                tokenize=False,
                add_generation_prompt=True,
            )
            for i, ex in enumerate(batch)
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False)
        enc = to_model_device(enc, model)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new = out[:, enc["input_ids"].shape[1] :]
        for i, text in enumerate(tokenizer.batch_decode(new, skip_special_tokens=True)):
            text = text.strip()
            results[start + i] = text if len(text) >= MIN_RESPONSE_CHARS else None
        print(f"  teacher {min(start + batch_size, len(examples))}/{len(examples)}", flush=True)
    return results


def make_sft_dataset(rows: list[dict], target_field: str) -> Dataset:
    ds_rows = []
    for row in rows:
        completion = str(row.get(target_field) or "").strip()
        if target_field == "sdft_response" and not completion:
            completion = str(row.get("response") or "").strip()
        if not completion:
            continue
        ds_rows.append({
            "prompt": [{"role": "user", "content": row["prompt"]}],
            "completion": [{"role": "assistant", "content": completion}],
        })
    return Dataset.from_list(ds_rows)


def attach_lora(model):
    peft_config = PeftLoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        task_type="CAUSAL_LM",
    )
    return get_peft_model(model, peft_config)


def train_lora_resident(
    model,
    tokenizer,
    rows: list[dict],
    *,
    target_field: str,
    output_dir: Path,
    max_steps: int | None = None,
    batch_size: int = TRAIN_BATCH_SIZE,
    num_epochs: float = 1.0,
):
    """LoRA SFT on the resident model. Does not unload afterward."""
    ds = make_sft_dataset(rows, target_field)
    if max_steps is None:
        # one epoch (or num_epochs) worth of optimizer steps at this batch size
        max_steps = max(1, int(round(num_epochs * len(ds) / max(batch_size, 1))))
    print(
        f"training {len(ds)} pairs → {output_dir} "
        f"(target={target_field}, steps={max_steps}, bs={batch_size})",
        flush=True,
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    model.config.use_cache = False
    if not isinstance(model, PeftModel):
        model = attach_lora(model)
    sft_config = SFTConfig(
        output_dir=str(output_dir),
        max_steps=max_steps,
        num_train_epochs=num_epochs,
        learning_rate=TRAIN_LR,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_length=TRAIN_MAX_LENGTH,
        warmup_steps=0,
        logging_steps=max(1, max_steps // 4),
        save_strategy="no",
        seed=SEED,
        report_to=[],
        completion_only_loss=True,
        use_cpu=not (
            torch.cuda.is_available()
            or (hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
        ),
        dataset_num_proc=1,
    )
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f"saved adapter → {output_dir}", flush=True)
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    return model


def train_lora_with_batch_backoff(
    model,
    tokenizer,
    rows: list[dict],
    *,
    target_field: str,
    output_dir: Path,
    batch_candidates: tuple[int, ...] = FULL_BATCH_CANDIDATES,
    start_batch: int = FULL_BATCH_SIZE,
    num_epochs: float = FULL_EPOCHS,
):
    """Train with max batch size; on OOM, drop bs and retry (reload LoRA on same base)."""
    candidates = [b for b in batch_candidates if b <= start_batch]
    if start_batch not in candidates:
        candidates = [start_batch] + candidates
    last_exc: BaseException | None = None
    for bs in candidates:
        try:
            # ensure clean LoRA attach if a prior OOM left a half-wrapped model
            if isinstance(model, PeftModel):
                model = unload_lora_to_base(model)
            max_steps = max(1, int(round(num_epochs * len(rows) / max(bs, 1))))
            return train_lora_resident(
                model,
                tokenizer,
                rows,
                target_field=target_field,
                output_dir=output_dir,
                max_steps=max_steps,
                batch_size=bs,
                num_epochs=num_epochs,
            )
        except Exception as exc:
            if _is_oom(exc) and bs != candidates[-1]:
                print(f"  train OOM at bs={bs} → retry smaller", flush=True)
                last_exc = exc
                release_gpu()
                if isinstance(model, PeftModel):
                    model = unload_lora_to_base(model)
                continue
            raise
    raise RuntimeError(f"train failed all batch sizes {candidates}") from last_exc


def unload_lora_to_base(model):
    """Drop LoRA adapters; return frozen base (same weights object when possible)."""
    if isinstance(model, PeftModel):
        base = model.unload()
        del model
        return base
    return model


def score_arm(gens: list[str], golds: list[str] | None = None) -> dict:
    golds = golds or [""] * len(gens)
    refusals = [bool(REFUSAL_RE.search(g)) for g in gens]
    rewards = [instruction_reward(g, golds[i] if i < len(golds) else "") for i, g in enumerate(gens)]
    return {
        "gens": gens,
        "refusals": refusals,
        "refusal_rate": sum(refusals) / max(len(gens), 1),
        "mean_reward": sum(rewards) / max(len(rewards), 1),
        "mean_chars": sum(len(g) for g in gens) / max(len(gens), 1),
        "rewards": rewards,
    }


def pairwise_wins(sdft: dict, other: dict, golds: list[str]) -> int:
    """Count prompts where SDFT beats `other` (non-refusal first, else higher reward)."""
    wins = 0
    for i in range(len(sdft["gens"])):
        sdft_ok = not sdft["refusals"][i]
        other_ok = not other["refusals"][i]
        if sdft_ok and not other_ok:
            wins += 1
        elif sdft_ok == other_ok:
            if sdft["rewards"][i] > other["rewards"][i]:
                wins += 1
    return wins


print("helpers ready")
print(
    f"MODEL={MODEL_NAME}  SEED={SEED}  STEP_COUNT={STEP_COUNT}  "
    f"batch={TRAIN_BATCH_SIZE}  lr={TRAIN_LR}  LoRA r/α={LORA_R}/{LORA_ALPHA}  shots={TEACHER_NUM_SHOTS}"
)
print(
    f"FULL stage: FULL_BATCH_SIZE={FULL_BATCH_SIZE}  FULL_MAX_ROWS={FULL_MAX_ROWS}  "
    f"FULL_EPOCHS={FULL_EPOCHS}  (remaining = data[{STEP_COUNT}:])"
)


## Data — seeded prefix slice

`yahma/alpaca-cleaned` shuffled with `SEED=13`, take `[:STEP_COUNT]` (`32`). Same prompts used for train **and** qualitative/heuristic eval.


In [ ]:
OUT_ROOT.mkdir(parents=True, exist_ok=True)

train_ex = load_demo_slice(STEP_COUNT, SEED)
assert len(train_ex) == STEP_COUNT, f"expected {STEP_COUNT} rows, got {len(train_ex)}"

demo_prompts = [e["prompt"] for e in train_ex]
demo_golds = [e["response"] for e in train_ex]
icl_demos = pick_icl_demos(train_ex, ICL_K)

with GOLD_PATH.open("w", encoding="utf-8") as fh:
    for e in train_ex:
        fh.write(
            json.dumps(
                {"prompt": e["prompt"], "response": e["response"], "sdft_response": e["response"]},
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"SEED={SEED}  STEP_COUNT={STEP_COUNT}  ICL demos={len(icl_demos)}")
print(f"wrote {GOLD_PATH}")
for i, e in enumerate(train_ex[:4]):
    print(f"  [{i}] {e['prompt'][:90].replace(chr(10), ' ')}…")


## One-load stage machine: base eval → teacher → SDFT train → SDFT eval → gold SFT

1. **Load base once**
2. Batched **ZS / ICL / CoT** on the demo prompts
3. **Teacher** SDFT targets (same base — no reload)
4. Attach LoRA → **train SDFT** (`batch_size=1`, `max_steps=STEP_COUNT`) → **eval SDFT** (same PeftModel)
5. `unload()` LoRA → attach fresh LoRA → **train gold SFT** → **eval gold SFT**
6. **Keep model resident** for the optional full remaining-data stage (do not release here)


In [ ]:
ARM_ORDER = ["zs", "icl", "cot", "sft_gold", "sdft"]
arm_results: dict = {}

print("loading base once…", flush=True)
tokenizer = load_tokenizer()
model = load_policy()

# --- base arms (resident) ---
print("generating zs / icl / cot (batched)…", flush=True)
zs = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="zs",
)
icl = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p, few_shots=icl_demos), label="icl",
)
cot = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p, cot=True), label="cot",
)
arm_results["zs"] = score_arm(zs, demo_golds)
arm_results["icl"] = score_arm(icl, demo_golds)
arm_results["cot"] = score_arm(cot, demo_golds)

# --- teacher on same base (no reload) ---
print("SDFT teacher pass (resident base)…", flush=True)
teacher_gens = teacher_generate_resident(model, tokenizer, train_ex)
sdft_rows = []
for ex, gen in zip(train_ex, teacher_gens):
    text = (gen or "").strip()
    if text:
        sdft_rows.append({"prompt": ex["prompt"], "response": ex["response"], "sdft_response": text})
with SDFT_PATH.open("w", encoding="utf-8") as fh:
    for row in sdft_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"wrote {SDFT_PATH}  non-empty={len(sdft_rows)}/{len(teacher_gens)}")
if not sdft_rows:
    raise RuntimeError("teacher pass produced zero non-empty sdft_response rows")

# --- attach LoRA, train SDFT, eval without unload ---
print("attach LoRA + train SDFT…", flush=True)
model = train_lora_resident(
    model, tokenizer, sdft_rows,
    target_field="sdft_response", output_dir=SDFT_ADAPTER, max_steps=STEP_COUNT,
)
print("eval SDFT (resident adapter)…", flush=True)
sdft_gens = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="sdft",
)
arm_results["sdft"] = score_arm(sdft_gens, demo_golds)

# --- unload LoRA → gold SFT train + eval on same base object ---
print("unload SDFT LoRA → train gold SFT…", flush=True)
model = unload_lora_to_base(model)
gold_rows = [{"prompt": e["prompt"], "response": e["response"]} for e in train_ex]
model = train_lora_resident(
    model, tokenizer, gold_rows,
    target_field="response", output_dir=SFT_ADAPTER, max_steps=STEP_COUNT,
)
print("eval gold SFT (resident adapter)…", flush=True)
sft_gens = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="sft_gold",
)
arm_results["sft_gold"] = score_arm(sft_gens, demo_golds)

# --- keep resident for optional full stage (release after that, or call release_gpu() if skipping) ---
print("demo stage machine done; model kept resident for optional full stage")


## Heuristic scoreboard + win counts

Primary metrics: **refusal rate**, **mean chars**, **instruction_reward**, and **SDFT pairwise wins** vs each baseline (no Qwen / AE2 judge).


In [ ]:
print("| arm | n | mean_reward | refusal_rate | mean_chars | sdft_wins_vs |")
print("|-----|---|-------------|--------------|------------|--------------|")
for name in ARM_ORDER:
    r = arm_results[name]
    wins = pairwise_wins(arm_results["sdft"], r, demo_golds) if name != "sdft" else "—"
    print(
        f"| {name} | {len(r['gens'])} | {r['mean_reward']:.3f} | "
        f"{r['refusal_rate']:.3f} | {r['mean_chars']:.0f} | {wins} |"
    )

baselines = ["zs", "icl", "cot", "sft_gold"]
total_wins = sum(pairwise_wins(arm_results["sdft"], arm_results[b], demo_golds) for b in baselines)
print()
print(f"SDFT pairwise win total vs baselines: {total_wins} / {STEP_COUNT * len(baselines)}")
print("metric = local heuristic (refusal + length + gold lexical overlap); not AE2 win_rate")


## Side-by-side qualitative table

Same demo prompt across all arms, plus the **gold** `output` / `response` from the training set. Prefers a row where ZS refuses and SDFT answers. **Full texts** (no truncation).


In [ ]:
import html
from IPython.display import HTML, display

idx, pick_reason = 0, "fallback index 0"
for i in range(len(demo_prompts)):
    if arm_results["zs"]["refusals"][i] and not arm_results["sdft"]["refusals"][i]:
        idx, pick_reason = i, "ZS refuses, SDFT answers"
        break
else:
    for i in range(len(demo_prompts)):
        if arm_results["sdft"]["rewards"][i] >= max(
            arm_results[a]["rewards"][i] for a in ("zs", "icl", "cot", "sft_gold")
        ):
            idx, pick_reason = i, "SDFT best heuristic reward on row"
            break

prompt_used = demo_prompts[idx]
gold_used = demo_golds[idx]
print(f"reason={pick_reason}")
print(f"prompt={prompt_used!r}")
print(f"gold_chars={len(gold_used)}")

cell_style = (
    "border:1px solid #ccc;padding:6px;vertical-align:top;"
    "white-space:pre-wrap;font-family:ui-monospace,monospace"
)
header = (
    "<h3>Same-prompt comparison (demo slice)</h3>"
    f"<p><b>Selection:</b> {html.escape(pick_reason)}</p>"
    f"<p><b>Prompt:</b> {html.escape(prompt_used)}</p>"
)
rows_html = [
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>",
    "<tr>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>arm</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>refusal</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>reward</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>output</th>"
    "</tr>",
    # Gold answer row (dataset output / response for this prompt)
    "<tr>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>gold</b></td>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>—</td>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>—</td>"
    f"<td style='{cell_style}'>{html.escape(gold_used)}</td>"
    "</tr>",
]
for arm in ARM_ORDER:
    r = arm_results[arm]
    rows_html.append(
        "<tr>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>{html.escape(arm)}</b></td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{'yes' if r['refusals'][idx] else 'no'}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{r['rewards'][idx]:.2f}</td>"
        f"<td style='{cell_style}'>{html.escape(r['gens'][idx])}</td>"
        "</tr>"
    )
rows_html.append("</table>")
display(HTML(header + "".join(rows_html)))


## Optional full AE2 / remaining-data stage

Train on the **remaining** seeded pool `data[STEP_COUNT:]` (same `SEED` shuffle as the demo; capped by `FULL_MAX_ROWS` for Colab time — set `None` for all remaining).

| Setting | Value |
|---------|--------|
| Slice | `data[STEP_COUNT : STEP_COUNT+FULL_MAX_ROWS]` |
| `FULL_BATCH_SIZE` | `16` (OOM backoff `32→16→8→…`) |
| Epochs | `FULL_EPOCHS=1` (`max_steps ≈ len / batch`) |
| Eval | Same demo prompts + heuristic scoreboard / full-text qualitative (no Qwen judge) |

Reuses the resident base from the demo when still loaded (unloads the gold-SFT LoRA first); otherwise reloads once. Skip these cells if you only want the few-step demo — then call `release_gpu()`.


In [ ]:
FULL_OUT.mkdir(parents=True, exist_ok=True)

full_train = load_remaining_slice(STEP_COUNT, SEED, FULL_MAX_ROWS)
assert full_train, "remaining slice empty — check STEP_COUNT / FULL_MAX_ROWS"
full_max_steps = max(1, int(round(FULL_EPOCHS * len(full_train) / max(FULL_BATCH_SIZE, 1))))

with FULL_GOLD_PATH.open("w", encoding="utf-8") as fh:
    for e in full_train:
        fh.write(
            json.dumps(
                {"prompt": e["prompt"], "response": e["response"], "sdft_response": e["response"]},
                ensure_ascii=False,
            )
            + "\n"
        )

print(
    f"FULL remaining: n={len(full_train)}  skip={STEP_COUNT}  "
    f"FULL_BATCH_SIZE={FULL_BATCH_SIZE}  FULL_EPOCHS={FULL_EPOCHS}  "
    f"~max_steps@{FULL_BATCH_SIZE}={full_max_steps}"
)
print(f"wrote {FULL_GOLD_PATH}")
for i, e in enumerate(full_train[:3]):
    print(f"  [{i}] {e['prompt'][:90].replace(chr(10), ' ')}…")


In [ ]:
# Reuse resident model from demo if available; otherwise reload once.
_need_reload = False
try:
    model  # noqa: F821
    tokenizer  # noqa: F821
    _ = next(model.parameters())
except Exception:
    _need_reload = True

if _need_reload:
    print("no resident model — loading base once for full stage…", flush=True)
    tokenizer = load_tokenizer()
    model = load_policy()
else:
    print("reusing resident model from demo stage…", flush=True)
    model = unload_lora_to_base(model)

full_arm_results: dict = {}

# Baseline ZS on demo prompts (same eval set as few-step stage)
print("FULL eval baseline zs (demo prompts)…", flush=True)
full_zs = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="full_zs",
)
full_arm_results["zs"] = score_arm(full_zs, demo_golds)

# Teacher on remaining train rows
print("FULL SDFT teacher pass…", flush=True)
full_teacher = teacher_generate_resident(
    model, tokenizer, full_train,
    num_shots=TEACHER_NUM_SHOTS,
    batch_size=FULL_TEACHER_BATCH,
)
full_sdft_rows = []
for ex, gen in zip(full_train, full_teacher):
    text = (gen or "").strip()
    if text:
        full_sdft_rows.append(
            {"prompt": ex["prompt"], "response": ex["response"], "sdft_response": text}
        )
with FULL_SDFT_PATH.open("w", encoding="utf-8") as fh:
    for row in full_sdft_rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"wrote {FULL_SDFT_PATH}  non-empty={len(full_sdft_rows)}/{len(full_teacher)}")
if not full_sdft_rows:
    raise RuntimeError("full teacher pass produced zero non-empty sdft_response rows")

# Train SDFT with max batch + OOM backoff
print("FULL train SDFT (max batch)…", flush=True)
model = train_lora_with_batch_backoff(
    model, tokenizer, full_sdft_rows,
    target_field="sdft_response",
    output_dir=FULL_SDFT_ADAPTER,
    batch_candidates=FULL_BATCH_CANDIDATES,
    start_batch=FULL_BATCH_SIZE,
    num_epochs=FULL_EPOCHS,
)
print("FULL eval SDFT…", flush=True)
full_sdft_gens = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="full_sdft",
)
full_arm_results["sdft"] = score_arm(full_sdft_gens, demo_golds)

# Unload → gold SFT on remaining
print("FULL unload → train gold SFT (max batch)…", flush=True)
model = unload_lora_to_base(model)
full_gold_rows = [{"prompt": e["prompt"], "response": e["response"]} for e in full_train]
model = train_lora_with_batch_backoff(
    model, tokenizer, full_gold_rows,
    target_field="response",
    output_dir=FULL_SFT_ADAPTER,
    batch_candidates=FULL_BATCH_CANDIDATES,
    start_batch=FULL_BATCH_SIZE,
    num_epochs=FULL_EPOCHS,
)
print("FULL eval gold SFT…", flush=True)
full_sft_gens = generate_batched(
    model, tokenizer, demo_prompts,
    messages_fn=lambda p: build_eval_messages(p), label="full_sft_gold",
)
full_arm_results["sft_gold"] = score_arm(full_sft_gens, demo_golds)

del model, tokenizer
release_gpu()
print("full remaining-data stage done; GPU released")


### Full-stage heuristic scoreboard

Same metrics as the demo (refusal / reward / chars / pairwise wins), evaluated on the **demo prompts** after training on the remaining slice.


In [ ]:
FULL_ARM_ORDER = ["zs", "sft_gold", "sdft"]

print("| arm | n | mean_reward | refusal_rate | mean_chars | sdft_wins_vs |")
print("|-----|---|-------------|--------------|------------|--------------|")
for name in FULL_ARM_ORDER:
    r = full_arm_results[name]
    wins = (
        pairwise_wins(full_arm_results["sdft"], r, demo_golds)
        if name != "sdft"
        else "—"
    )
    print(
        f"| {name} | {len(r['gens'])} | {r['mean_reward']:.3f} | "
        f"{r['refusal_rate']:.3f} | {r['mean_chars']:.0f} | {wins} |"
    )

full_baselines = ["zs", "sft_gold"]
full_total_wins = sum(
    pairwise_wins(full_arm_results["sdft"], full_arm_results[b], demo_golds)
    for b in full_baselines
)
print()
print(
    f"FULL SDFT pairwise win total vs baselines: "
    f"{full_total_wins} / {STEP_COUNT * len(full_baselines)}"
)
print("metric = local heuristic; not AE2 win_rate / Qwen judge")


### Full-stage qualitative (gold + arms, full text)


In [ ]:
import html
from IPython.display import HTML, display

f_idx, f_reason = 0, "fallback index 0"
for i in range(len(demo_prompts)):
    if full_arm_results["zs"]["refusals"][i] and not full_arm_results["sdft"]["refusals"][i]:
        f_idx, f_reason = i, "ZS refuses, FULL SDFT answers"
        break
else:
    for i in range(len(demo_prompts)):
        if full_arm_results["sdft"]["rewards"][i] >= max(
            full_arm_results[a]["rewards"][i] for a in ("zs", "sft_gold")
        ):
            f_idx, f_reason = i, "FULL SDFT best heuristic reward on row"
            break

f_prompt = demo_prompts[f_idx]
f_gold = demo_golds[f_idx]
print(f"reason={f_reason}")
print(f"prompt={f_prompt!r}")

cell_style = (
    "border:1px solid #ccc;padding:6px;vertical-align:top;"
    "white-space:pre-wrap;font-family:ui-monospace,monospace"
)
header = (
    "<h3>Same-prompt comparison (after full remaining-data stage)</h3>"
    f"<p><b>Selection:</b> {html.escape(f_reason)}</p>"
    f"<p><b>Prompt:</b> {html.escape(f_prompt)}</p>"
)
rows_html = [
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>",
    "<tr>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>arm</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>refusal</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>reward</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>output</th>"
    "</tr>",
    "<tr>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>gold</b></td>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>—</td>"
    "<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>—</td>"
    f"<td style='{cell_style}'>{html.escape(f_gold)}</td>"
    "</tr>",
]
for arm in FULL_ARM_ORDER:
    r = full_arm_results[arm]
    rows_html.append(
        "<tr>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>{html.escape(arm)}</b></td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{'yes' if r['refusals'][f_idx] else 'no'}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{r['rewards'][f_idx]:.2f}</td>"
        f"<td style='{cell_style}'>{html.escape(r['gens'][f_idx])}</td>"
        "</tr>"
    )
rows_html.append("</table>")
display(HTML(header + "".join(rows_html)))


## What to try next

1. **Change the demo slice** — edit `SEED` / `STEP_COUNT` (keep `TRAIN_BATCH_SIZE=1`).
2. **Widen the full stage** — raise `FULL_MAX_ROWS` (or set `None` for all remaining) / tweak `FULL_BATCH_SIZE`.
3. **Full toolkit** — [Local-SDFT](https://github.com/lin826/Local-SDFT) for `/perf` chat, GRPO, CLI, and optional Qwen AE2 judge scripts.

Happy tinkering — 230M + few-step demo is small enough that curiosity is the bottleneck, not VRAM.
